In [ ]:
import joblib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# =========================================================
# 1. LOAD DATASET & TRAIN MODEL
# =========================================================
print("Loading Training.csv...")
df = pd.read_csv("Training.csv")

# Clean artifact column if present
if "Unnamed: 133" in df.columns:
    df.drop("Unnamed: 133", axis=1, inplace=True)

# Separate features (symptoms) and target (prognosis)
X = df.drop("prognosis", axis=1)
y = df["prognosis"]


# Encode target disease names
le = LabelEncoder()
y_encoded = le.fit_transform(y)


# Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

# Initialize and Train Random Forest
print("Training Random Forest Classifier...")
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
)
rf.fit(X_train, y_train)

# Evaluate Accuracy
y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {acc * 100:.2f}%")

# Save Models for Spring Boot / Deployment
joblib.dump(rf, "disease_model.pkl")
joblib.dump(le, "label_encoder.pkl")
print("Model files saved successfully as 'disease_model.pkl' and 'label_encoder.pkl'.\n")

# =========================================================
# 2. INITIALIZE NLP EMBEDDING ENGINE
# =========================================================
print("Loading NLP SentenceTransformer model (all-MiniLM-L6-v2)...")
nlp_model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract feature names (132 symptoms)
feature_columns = list(X.columns)

# Convert column format from snake_case to natural readable text ('yellow_crust_ooze' -> 'yellow crust ooze')
formatted_columns = [col.replace("_", " ") for col in feature_columns]

# Pre-compute vector embeddings for all 132 symptom columns
column_embeddings = nlp_model.encode(formatted_columns, convert_to_tensor=True)


# =========================================================
# 3. RAW SENTENCE TO COLUMN MAPPER & PREDICTOR
# =========================================================
def predict_from_raw_text(
    raw_user_text: str, similarity_threshold: float = 0.45
):
    """Parses a natural sentence, maps phrases to exact dataset columns using cosine

    similarity, and predicts the disease.
    """
    # Tokenize input into clauses/phrases
    delimiters = [",", " and ", " with ", ".", " i have ", " suffering from "]
    phrases = [raw_user_text.lower()]
    for d in delimiters:
        phrases = [sub for p in phrases for sub in p.split(d)]

    phrases = [p.strip() for p in phrases if p.strip()]

    # Extract matching column names
    detected_symptoms = set()
    for phrase in phrases:
        phrase_embedding = nlp_model.encode(phrase, convert_to_tensor=True)

        # Calculate cosine similarity scores
        cosine_scores = util.cos_sim(phrase_embedding, column_embeddings)[0]

        # Filter symptoms meeting or exceeding the threshold score
        matched_indices = np.where(
            cosine_scores.cpu().numpy() >= similarity_threshold
        )[0]
        for idx in matched_indices:
            detected_symptoms.add(feature_columns[idx])

    detected_symptoms = list(detected_symptoms)

    # Build feature input vector (1s for matched symptoms, 0s for others)
    input_vector = pd.DataFrame([[0] * len(X.columns)], columns=X.columns)
    for symptom in detected_symptoms:
        input_vector[symptom] = 1

    # Predict disease
    prediction = rf.predict(input_vector)
    confidence = float(max(rf.predict_proba(input_vector)[0]))
    predicted_disease = le.inverse_transform(prediction)[0]

    return {
        "user_input": raw_user_text,
        "matched_columns": detected_symptoms,
        "predicted_disease": predicted_disease,
        "confidence_score": f"{round(confidence * 100, 2)}%",
    }


# =========================================================
# 4. TEST EXAMPLE
# =========================================================
if __name__ == "__main__":
    sample_sentence = "high fever"

    print("--- Test Run ---")
    result = predict_from_raw_text(sample_sentence)

    print(f"User Input       : {result['user_input']}")
    print(f"Matched Columns  : {result['matched_columns']}")
    print(f"Predicted Disease: {result['predicted_disease']}")
    print(f"Confidence       : {result['confidence_score']}")